# Chronos

In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt

from chronos import BaseChronosPipeline
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

/Users/federicosabbadini/Library/Mobile Documents/com~apple~CloudDocs/Workspaces/gitHub-workspace/patchAliasing/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from chronos import BaseChronosPipeline, Chronos2Pipeline

pipeline = BaseChronosPipeline.from_pretrained(
    "amazon/chronos-t5-small",
    device_map="cpu",
)

In [3]:
print(dir(pipeline))

['__annotations__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_prepare_and_validate_context', 'dtypes', 'embed', 'forecast_type', 'from_pretrained', 'inner_model', 'model', 'model_context_length', 'model_prediction_length', 'predict', 'predict_df', 'predict_fev', 'predict_quantiles', 'tokenizer']


In [4]:
import torch
import pprint

def inspect_chronos_pipeline(pipeline):
    print("\n================ PIPELINE TYPE ================\n")
    print(type(pipeline))

    print("\n================ TOP-LEVEL ATTRIBUTES ================\n")
    attrs = [a for a in dir(pipeline) if not a.startswith("_")]
    print(attrs)

    print("\n================ KEY ATTRIBUTES (IF PRESENT) ================\n")
    keys = [
        "model_prediction_length",
        "model_context_length",
        "forecast_type",
        "inner_model",
        "model",
        "quantiles",
    ]

    for k in keys:
        if hasattr(pipeline, k):
            print(f"{k}: {getattr(pipeline, k)}")
        else:
            print(f"{k}: 1")

    print("\n================ INNER MODEL INFO ================\n")
    if hasattr(pipeline, "inner_model"):
        im = pipeline.inner_model
        print("inner_model type:", type(im))

        # try config
        if hasattr(im, "config"):
            print("\ninner_model.config:")
            pprint.pprint(im.config)

        # try attributes of inner model
        print("\ninner_model attributes:")
        print([a for a in dir(im) if not a.startswith("_")][:50])
    else:
        print("No inner_model")

    print("\n================ PREDICT METHOD SIGNATURE ================\n")
    try:
        import inspect
        print(inspect.signature(pipeline.predict))
    except Exception as e:
        print("Could not inspect predict:", e)

    print("\n================ DEVICE INFO ================\n")
    try:
        p = next(pipeline.inner_model.parameters())
        print("device:", p.device)
        print("dtype:", p.dtype)
    except Exception:
        print("No torch parameters found")

    print("\n================ QUICK SANITY CHECK ================\n")
    try:
        print("model_prediction_length:", getattr(pipeline, "model_prediction_length", None))
        print("context_length:", getattr(pipeline, "model_context_length", None))
    except Exception as e:
        print("error:", e)

inspect_chronos_pipeline(pipeline)


================ PIPELINE TYPE ================

<class 'chronos.chronos.ChronosPipeline'>

================ TOP-LEVEL ATTRIBUTES ================

['dtypes', 'embed', 'forecast_type', 'from_pretrained', 'inner_model', 'model', 'model_context_length', 'model_prediction_length', 'predict', 'predict_df', 'predict_fev', 'predict_quantiles', 'tokenizer']

================ KEY ATTRIBUTES (IF PRESENT) ================

model_prediction_length: 64
model_context_length: 512
forecast_type: ForecastType.SAMPLES
inner_model: T5ForConditionalGeneration(
  (shared): Embedding(4096, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(4096, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=

## Model

In [5]:
model = pipeline.model

print(type(model))

<class 'chronos.chronos.ChronosModel'>


In [6]:
print(pipeline.model.config)

ChronosConfig(tokenizer_class='MeanScaleUniformBins', tokenizer_kwargs={'low_limit': -15.0, 'high_limit': 15.0}, context_length=512, prediction_length=64, n_tokens=4096, n_special_tokens=2, pad_token_id=0, eos_token_id=1, use_eos_token=True, model_type='seq2seq', num_samples=20, temperature=1.0, top_k=50, top_p=1.0)


### Layers

In [7]:
pre_transformer_modules = []
encoder_modules = []
decoder_modules = []
post_transformer_modules = []

modules = []
for name, module in model.named_modules():
    modules.append((name, module))

In [8]:
for name, module in modules:
    if "encoder" not in name:
        pre_transformer_modules.append((name, module))
        modules.remove((name, module))
    else:
        break

i = 0
for name, module in pre_transformer_modules:
    if i == 0:
        print(f"Pre-transformer modules: {type(module)}")
    else:
        print(f"{i}: {name}: {type(module)}")
    i += 1

Pre-transformer modules: <class 'chronos.chronos.ChronosModel'>
1: model.shared: <class 'torch.nn.modules.sparse.Embedding'>


In [9]:
for name, module in modules:
    if "encoder" in name and "decoder" not in name:
        encoder_modules.append((name, module))
        modules.remove((name, module))
        
i = 0
for name, module in encoder_modules:
    if i == 0:
        print(f"Encoder modules: {type(module)}")
    else:
        print(f"{i}: {name}: {type(module)}")
    i += 1

Encoder modules: <class 'transformers.models.t5.modeling_t5.T5Stack'>
1: model.encoder.block.0: <class 'transformers.models.t5.modeling_t5.T5Block'>
2: model.encoder.block.0.layer.0: <class 'transformers.models.t5.modeling_t5.T5LayerSelfAttention'>
3: model.encoder.block.0.layer.0.SelfAttention.q: <class 'torch.nn.modules.linear.Linear'>
4: model.encoder.block.0.layer.0.SelfAttention.v: <class 'torch.nn.modules.linear.Linear'>
5: model.encoder.block.0.layer.0.SelfAttention.relative_attention_bias: <class 'torch.nn.modules.sparse.Embedding'>
6: model.encoder.block.0.layer.0.dropout: <class 'torch.nn.modules.dropout.Dropout'>
7: model.encoder.block.0.layer.1.DenseReluDense: <class 'transformers.models.t5.modeling_t5.T5DenseActDense'>
8: model.encoder.block.0.layer.1.DenseReluDense.wo: <class 'torch.nn.modules.linear.Linear'>
9: model.encoder.block.0.layer.1.DenseReluDense.act: <class 'torch.nn.modules.activation.ReLU'>
10: model.encoder.block.0.layer.1.dropout: <class 'torch.nn.modules.d

In [10]:
for name, module in modules:
    if "decoder" in name:
        decoder_modules.append((name, module))
        modules.remove((name, module))

i = 0
for name, module in decoder_modules:
    if i == 0:
        print(f"Decoder modules: {type(module)}")
    else:
        print(f"{i}: {name}: {type(module)}")
    i += 1

Decoder modules: <class 'transformers.models.t5.modeling_t5.T5Stack'>
1: model.decoder.block.0: <class 'transformers.models.t5.modeling_t5.T5Block'>
2: model.decoder.block.0.layer.0: <class 'transformers.models.t5.modeling_t5.T5LayerSelfAttention'>
3: model.decoder.block.0.layer.0.SelfAttention.q: <class 'torch.nn.modules.linear.Linear'>
4: model.decoder.block.0.layer.0.SelfAttention.v: <class 'torch.nn.modules.linear.Linear'>
5: model.decoder.block.0.layer.0.SelfAttention.relative_attention_bias: <class 'torch.nn.modules.sparse.Embedding'>
6: model.decoder.block.0.layer.0.dropout: <class 'torch.nn.modules.dropout.Dropout'>
7: model.decoder.block.0.layer.1.EncDecAttention: <class 'transformers.models.t5.modeling_t5.T5Attention'>
8: model.decoder.block.0.layer.1.EncDecAttention.k: <class 'torch.nn.modules.linear.Linear'>
9: model.decoder.block.0.layer.1.EncDecAttention.o: <class 'torch.nn.modules.linear.Linear'>
10: model.decoder.block.0.layer.1.dropout: <class 'torch.nn.modules.dropout

In [11]:
for name, module in modules[20:]:
    if "encoder" not in name and "decoder" not in name:
        post_transformer_modules.append((name, module))

i = 0
for name, module in post_transformer_modules:
    if i == 0:
        print(f"Post-transformer modules: {type(module)}")
    else:
        print(f"{i}: {name}: {type(module)}")
    i += 1

Post-transformer modules: <class 'torch.nn.modules.linear.Linear'>


## Embeddings Learning

In [12]:
captured = {
    "patches": None, # This will be a list of tensors, one for each patch, containing the raw pixel values of the patch.
    "patch_embeddings": None, # This will be a list of tensors, one for each patch, containing the embeddings of the patches after the patch embedding layer.
    "encoder_inputs": None, # This will be a list of tensors, one for each patch, containing the inputs to the encoder after the positional encoding and patch embedding layers.
}

### Hooks

In [13]:
# Hook is a function that will be called every time the module/layer is called. 
# Using PyTorch forward, hooks to “spy on” intermediate values inside a neural network while it runs.


# Dictionary to store captured tensors
captured = {}
# List to keep hook handles
hooks = []

\
def to_cpu(x):
    """
    Utility function to move tensors to CPU and detach them from the computation graph.
    This is useful for storing intermediate values without keeping the entire computation graph in memory.
    """
    if torch.is_tensor(x):
        # Detach the tensor from the computation graph and move it to CPU
        return x.detach().cpu()
    elif isinstance(x, tuple):
            # If the input is a tuple, apply to_cpu to each element in the tuple and return a new tuple with the results
            return tuple(to_cpu(item) for item in x)
    else:  
        # If the input is neither a tensor nor a tuple, return it as is (e.g., for non-tensor inputs)
        return x

#### 1. Embeddings Hook

With simple chrons i have only 1 layer of preprocessing: 
- model.shared: <class 'torch.nn.modules.sparse.Embedding'> so


In [14]:
def embedding_hook(module, inputs, outputs):
    captured["embedding"] = to_cpu(outputs)

In [15]:
# we have to access to 1: model.encoder.block.0: <class 'transformers.models.t5.modeling_t5.T5Block'>

hooks.append(
    pipeline.model.model.shared.register_forward_hook(embedding_hook)
)

### Input

In [16]:
series1 = torch.tensor([
    11.0, 1.2, 1.4, 1.1,
    0.9, 1.3, 1.5, 1.7,
    1.8, 1.6, 1.4, 1.2,
    1.1, 1.0, 0.8, 0.7,
    0.9, 1.1, 1.3, 1.5,
    1.7, 1.9, 2.0, 1.8,
    1.6, 1.4, 1.2, 1.0,
], dtype=torch.float32)

In [17]:
series2 = torch.tensor([
    0.5, 0.6, 0.7, 0.8,
    0.9, 1.0, 1.1, 1.2,
    1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.0, 2.1, 2.2, 2.3,
    2.4, 2.5, 2.6, 2.7, 2.8, 2.9, 3.0, 3.1,
], dtype=torch.float32)

In [18]:
series3 = torch.tensor([
    2.0, 1.9, 1.8, 1.7,
    1.6, 1.5, 1.4, 1.3,
    1.2, 1.1, 1.0, 0.9, 0.8, 0.7, 0.6, 0.5, 0.4, 0.3, 0.2,  
    0.1, 0.0, -0.1, -0.2, -0.3, -0.4,
], dtype=torch.float32)

In [19]:
input = series1 

print("Input series shape:", input.shape)
# We are UNIVARIATE, so we have only one feature dimension. If the model expects a feature dimension, we need to add it.
# input = input.unsqueeze(-1)  # Add a feature dimension to the input series
# print("Input series shape after adding feature dimension:", input.shape)
input = input.unsqueeze(0)  # Add batch dimension
print("Input series shape after adding batch dimension:", input.shape)

Input series shape: torch.Size([28])
Input series shape after adding batch dimension: torch.Size([1, 28])


### Processing

In [20]:
with torch.no_grad():

    with torch.no_grad():
        forecast = pipeline.predict(
            input,
            prediction_length=11,
            num_samples=1,
        )

In [21]:
embedding = captured.get("embedding", None)

remember = {}
def print_remember():
    for key, value in remember.items():
        if value is not None and key is not None:
            print(f"REMEMBER -- {key}: {value}")

In [22]:
print("\n================================================")
print("0. INPUT SERIES")
print("================================================")

input_shape = input.shape
print("Shape:", input_shape)

for i in range(input_shape[1]):
    print(f"Value {i}: {input[0, i].item()}")
    
remember["Input shape"] = input_shape


0. INPUT SERIES
Shape: torch.Size([1, 28])
Value 0: 11.0
Value 1: 1.2000000476837158
Value 2: 1.399999976158142
Value 3: 1.100000023841858
Value 4: 0.8999999761581421
Value 5: 1.2999999523162842
Value 6: 1.5
Value 7: 1.7000000476837158
Value 8: 1.7999999523162842
Value 9: 1.600000023841858
Value 10: 1.399999976158142
Value 11: 1.2000000476837158
Value 12: 1.100000023841858
Value 13: 1.0
Value 14: 0.800000011920929
Value 15: 0.699999988079071
Value 16: 0.8999999761581421
Value 17: 1.100000023841858
Value 18: 1.2999999523162842
Value 19: 1.5
Value 20: 1.7000000476837158
Value 21: 1.899999976158142
Value 22: 2.0
Value 23: 1.7999999523162842
Value 24: 1.600000023841858
Value 25: 1.399999976158142
Value 26: 1.2000000476837158
Value 27: 1.0


In [23]:
print_remember()
print("\n================================================")
print("1. EMBEDDING MODULE OUTPUT")
print("================================================")

embedding_shape = embedding.shape if embedding is not None else None
print("Shape:", embedding_shape)

for i in range(embedding_shape[1]):
    print(embedding[0, i])

remember["Embedding shape"] = embedding_shape

REMEMBER -- Input shape: torch.Size([1, 28])

1. EMBEDDING MODULE OUTPUT
Shape: torch.Size([1, 1, 512])
tensor([ 1.2293e+00,  5.5556e-01, -2.1027e-01, -4.5025e-01,  7.5969e-01,
         6.5943e-01, -3.5164e-01, -8.6897e-01,  4.8303e-01,  1.8332e-01,
         4.2428e-01,  4.2627e-01, -9.6606e-01,  1.0989e+00, -3.7933e-01,
         6.2209e-01,  2.1947e-01, -1.0311e-01, -8.4050e-01, -7.4154e-01,
         1.4469e-01,  4.7950e-01,  6.8503e-01,  6.3349e-01, -9.3140e-02,
         6.9794e-01, -5.5181e-01,  2.4441e-01,  5.3048e-01,  5.4488e-01,
         2.4447e-01,  1.1129e-03,  5.1196e-01,  2.2066e-01,  9.6887e-01,
        -6.5641e-01,  8.7104e-01, -4.4134e-01,  4.0274e-02,  5.6302e-01,
         9.7068e-01,  1.0374e+00,  3.8943e-01, -7.3237e-01,  4.1561e-01,
        -6.1076e-01, -5.1303e-01,  5.9042e-01, -3.9820e-01, -1.1201e+00,
         2.0288e-01, -7.2002e-01,  2.6832e-01,  1.8279e-01, -4.9179e-01,
         7.3830e-01,  1.0476e+00, -7.3111e-01, -3.2470e-02, -3.7191e-01,
        -4.7074e-01,

In [24]:
print_remember()
print("\n================================================")
print("FORECAST")
print("================================================")

forecast_shape = forecast.shape
print("Shape:", forecast_shape)
# the forecast shape is (batch_size, prediction_length, num_features), 
# where num_features is 1 for univariate forecasting. 
# So we can print the forecast as a list of values for each time step in the forecast horizon.

print("\nForecast:" )
print(forecast)

REMEMBER -- Input shape: torch.Size([1, 28])
REMEMBER -- Embedding shape: torch.Size([1, 1, 512])

FORECAST
Shape: torch.Size([1, 1, 11])

Forecast:
tensor([[[1.1962, 1.6032, 1.6032, 1.6032, 1.1962, 1.0976, 0.9003, 0.9989,
          1.2949, 1.6032, 1.6032]]])
